# Interpolation and Data Cleaning Workshop: Solved Version

This solved notebook keeps the interpolation and data-cleaning workshop structure, with one reference implementation. It uses real Neon recordings only.

The style is inspired by coding-test notebooks: complete a small function, run the public tests, then inspect the result visually on real data.

This solved copy keeps the same flow as the student notebook, but the exercise cells contain one possible solution.


## Learning Goals

By the end, you should be able to:

1. Build a boolean mask for a blink or held-out gap.
2. Interpolate missing samples using local time anchors.
3. Score interpolation quality on a held-out segment.
4. Compare interpolation methods on real pupil data.
5. Change filtering and interpolation parameters and inspect the effect.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from nb_setup import DEFAULT_PARTICIPANT_ID, ensure_workshop_data, setup
PROJECT_ROOT = setup()
ensure_workshop_data(PROJECT_ROOT)

from libs.analysis.recording_helpers import find_neon_recording, load_neon, neon_participant_id
from libs.analysis.preprocessing import PreprocessingPipeline

METHODS = ["linear", "pchip", "nearest", "weighted_average"]

pipe = PreprocessingPipeline()
print("Available methods:", ", ".join(METHODS))

Project root: /Users/eduardo/Workspace/Unity/UnityProjects/IMRF_2026_SpatialAudio_Eyetracking/IMRF_Eyetracking
Workshop data ready: /Users/eduardo/Workspace/Unity/UnityProjects/IMRF_2026_SpatialAudio_Eyetracking/IMRF_Eyetracking/data_output
Available methods: linear, pchip, nearest, weighted_average, cubic


## Exercise 2 — Build a Gap Mask

A mask is a boolean array: `True` for samples we want to hide or repair, `False` for samples we keep.

Complete `make_gap_mask`. The public test uses a tiny time vector so you can reason about the expected answer by hand.


In [2]:
def make_gap_mask(times: np.ndarray, center_s: float, duration_s: float) -> np.ndarray:
    """Return a boolean mask for samples inside a centered time window."""
    times = np.asarray(times, dtype=float)
    half_duration = duration_s / 2.0
    start_s = center_s - half_duration
    end_s = center_s + half_duration
    return (times >= start_s) & (times <= end_s)


In [3]:
def test_make_gap_mask(fn):
    times_test = np.array([0.0, 0.1, 0.2, 0.3, 0.4, 0.5])
    expected = np.array([False, False, True, True, False, False])
    out = fn(times_test, center_s=0.25, duration_s=0.20)

    assert isinstance(out, np.ndarray), "Return a numpy array."
    assert out.dtype == bool, "The mask must be boolean."
    assert out.shape == times_test.shape, "Mask shape must match times shape."
    assert np.array_equal(out, expected), f"Expected {expected}, got {out}."
    print("Exercise 2 passed.")


test_make_gap_mask(make_gap_mask)


Exercise 3 passed.


## Exercise 3 — Linear Interpolation

Now complete a simple linear interpolator. The function receives the original signal and a mask. Pretend the masked samples are missing, then estimate them from surrounding visible samples.

This is intentionally simpler than the production pipeline. The production version handles edge cases and multiple contiguous gaps more carefully.


In [4]:
def linear_interpolate_student(
    times: np.ndarray,
    signal: np.ndarray,
    mask: np.ndarray,
) -> np.ndarray:
    """Linearly interpolate masked samples and preserve visible samples."""
    times = np.asarray(times, dtype=float)
    out = np.asarray(signal, dtype=float).copy()
    mask = np.asarray(mask, dtype=bool)

    valid = (~mask) & np.isfinite(times) & np.isfinite(out)
    if valid.sum() < 2:
        return out

    interpolated = np.interp(times, times[valid], out[valid])
    out[mask] = interpolated[mask]
    return out


In [5]:
def test_linear_interpolate_student(fn):
    times_test = np.array([0, 1, 2, 3, 4], dtype=float)
    signal_test = np.array([0, 2, 99, 99, 8], dtype=float)
    mask_test = np.array([False, False, True, True, False])
    expected = np.array([0, 2, 4, 6, 8], dtype=float)

    out = fn(times_test, signal_test, mask_test)
    assert isinstance(out, np.ndarray), "Return a numpy array."
    assert out.shape == signal_test.shape, "Output shape must match signal shape."
    assert np.allclose(out, expected), f"Expected {expected}, got {out}."
    assert np.allclose(out[~mask_test], signal_test[~mask_test]), "Visible samples should not change."
    print("Exercise 3 passed.")


test_linear_interpolate_student(linear_interpolate_student)


Exercise 4 passed.


## Exercise 4 — Evaluate the Reconstructed Gap

When we hide a segment where the true signal is known, we can score each method only inside the hidden gap.

Complete `score_gap` so it returns the number of evaluated samples, MAE, and RMSE.


In [6]:
def score_gap(
    true_signal: np.ndarray,
    predicted_signal: np.ndarray,
    mask: np.ndarray,
) -> dict:
    """Compute interpolation error inside the masked region."""
    true_signal = np.asarray(true_signal, dtype=float)
    predicted_signal = np.asarray(predicted_signal, dtype=float)
    mask = np.asarray(mask, dtype=bool)

    err = predicted_signal[mask] - true_signal[mask]
    err = err[np.isfinite(err)]

    if err.size == 0:
        return {"n": 0, "mae": np.nan, "rmse": np.nan}

    return {
        "n": int(err.size),
        "mae": float(np.mean(np.abs(err))),
        "rmse": float(np.sqrt(np.mean(err ** 2))),
    }


In [7]:
def test_score_gap(fn):
    true_signal = np.array([1, 2, 3, 4, 5], dtype=float)
    predicted_signal = np.array([1, 2, 2, 6, 5], dtype=float)
    mask = np.array([False, False, True, True, False])

    out = fn(true_signal, predicted_signal, mask)
    assert set(out) == {"n", "mae", "rmse"}, "Return keys must be exactly n, mae, rmse."
    assert out["n"] == 2, "There are two masked samples."
    assert np.isclose(out["mae"], 1.5), f"Unexpected MAE: {out['mae']}"
    assert np.isclose(out["rmse"], np.sqrt(2.5)), f"Unexpected RMSE: {out['rmse']}"
    print("Exercise 4 passed.")


test_score_gap(score_gap)


Exercise 5 passed.


## Load One Real Neon Recording

This notebook intentionally uses real data only. By default it loads `DEFAULT_PARTICIPANT_ID` from `nb_setup.py` (`p0097`).

Change `PARTICIPANT_ID` to `"p0096"` or `"p0097"`, or set `PREFERRED_RECORDING_ID` to a Neon folder UUID.


In [8]:
EXPERIMENT = "IMRFSpatialAV"
PARTICIPANT_ID = DEFAULT_PARTICIPANT_ID  # options: "p0096", "p0097", "p0099"
PREFERRED_RECORDING_ID = None # optional Neon folder UUID override
# In these Neon exports, pupil diameter columns are usually named
# diameter_left and diameter_right. Older scripts may call similar
# signals pupil_left/pupil_right, so we print the actual columns below.
SIGNAL_COLUMN = "diameter_left"

HELDOUT_GAP_S = 0.20
ZOOM_PAD_S = 0.55
N_NATIVE_BLINKS = 1

NEON_ROOT = PROJECT_ROOT / "data_output" / EXPERIMENT / "neon"
RECORDING_PATH = find_neon_recording(
    NEON_ROOT,
    participant_id=PARTICIPANT_ID,
    recording_id=PREFERRED_RECORDING_ID,
)
if RECORDING_PATH is None or not RECORDING_PATH.is_dir():
    raise FileNotFoundError(
        f"No Neon recording found under {NEON_ROOT} for participant {PARTICIPANT_ID!r}. "
        "Check PARTICIPANT_ID or set PREFERRED_RECORDING_ID."
    )

SELECTED_PARTICIPANT = neon_participant_id(RECORDING_PATH) or PARTICIPANT_ID
rec = load_neon(str(RECORDING_PATH))
if rec.pupil is None or rec.pupil.empty:
    raise ValueError("No pupil stream was loaded for this recording.")

source_df = rec.pupil.copy()
print("Pupil columns:", list(source_df.columns))
if SIGNAL_COLUMN not in source_df.columns:
    raise ValueError(f"Column {SIGNAL_COLUMN!r} not found. Available columns: {list(source_df.columns)}")

raw_times = source_df["timestamp"].to_numpy(dtype=float)
raw_signal = source_df[SIGNAL_COLUMN].to_numpy(dtype=float)
valid = np.isfinite(raw_times) & np.isfinite(raw_signal)
times = raw_times[valid]
signal = raw_signal[valid]
signal_col = SIGNAL_COLUMN

blinks = rec.blinks.copy() if rec.blinks is not None else pd.DataFrame(columns=["start_timestamp", "end_timestamp"])

print(f"Participant: {SELECTED_PARTICIPANT}")
print(f"Recording : {RECORDING_PATH.name}")
print(f"Signal    : {signal_col}")
print(f"Samples   : {len(signal):,}")
print(f"Duration  : {times[-1] - times[0]:.1f} s")
print(f"Blinks    : {len(blinks):,}")

display(source_df.head())
if not blinks.empty:
    display(blinks.head())


Pupil columns: ['timestamp', 'diameter_left', 'diameter_right']
Participant: p0097
Recording : fdd1a874-8df3-4c46-b994-51272f98ec13
Signal    : diameter_left
Samples   : 40,934
Duration  : 205.1 s
Blinks    : 10


,timestamp,diameter_left,diameter_right
0,1.782119e+09,4.736677,4.321436
1,1.782119e+09,4.766074,4.328786
2,1.782119e+09,4.755050,4.303063
3,1.782119e+09,4.733002,4.321436
4,1.782119e+09,4.725653,4.277340


,timestamp,start_timestamp,end_timestamp
0,1.782119e+09,1.782119e+09,1.782119e+09
1,1.782119e+09,1.782119e+09,1.782119e+09
2,1.782119e+09,1.782119e+09,1.782119e+09
3,1.782119e+09,1.782119e+09,1.782119e+09
4,1.782119e+09,1.782119e+09,1.782119e+09


## Helper Functions for the Real-Data Sections

These helpers use the repository's `PreprocessingPipeline`. Students do not need to edit this cell.


In [9]:
def mask_to_blinks_df(mask: np.ndarray, times: np.ndarray) -> pd.DataFrame:
    m = np.asarray(mask, dtype=bool).astype(int)
    d = np.diff(np.concatenate(([0], m, [0])))
    starts = np.where(d == 1)[0]
    ends = np.where(d == -1)[0] - 1
    rows = [
        {"start_timestamp": float(times[s]), "end_timestamp": float(times[e])}
        for s, e in zip(starts, ends)
    ]
    return pd.DataFrame(rows, columns=["start_timestamp", "end_timestamp"])


def native_blink_mask(times: np.ndarray, blinks: pd.DataFrame, guard_s: float = 0.0) -> np.ndarray:
    mask = np.zeros(len(times), dtype=bool)
    if blinks is None or blinks.empty:
        return mask

    for _, blink in blinks.iterrows():
        start = blink.get("start_timestamp", blink.get("timestamp", np.nan))
        end = blink.get("end_timestamp", start)
        if not np.isfinite(start):
            continue
        if not np.isfinite(end) or end <= start:
            end = start + 0.15
        mask |= (times >= start - guard_s) & (times <= end + guard_s)
    return mask


def choose_heldout_mask(
    times: np.ndarray,
    blinks: pd.DataFrame,
    duration_s: float,
    guard_s: float = 0.25,
) -> np.ndarray:
    dt = float(np.median(np.diff(times))) if len(times) > 1 else 1 / 200.0
    gap_n = max(3, int(round(duration_s / dt)))
    avoid = native_blink_mask(times, blinks, guard_s=guard_s)
    centers = np.argsort(np.abs(np.arange(len(times)) - len(times) // 2))

    for center in centers:
        start = int(center) - gap_n // 2
        end = start + gap_n
        if start < 1 or end >= len(times) - 1:
            continue
        if not avoid[start:end].any():
            mask = np.zeros(len(times), dtype=bool)
            mask[start:end] = True
            return mask

    start = max(1, len(times) // 2 - gap_n // 2)
    end = min(len(times) - 1, start + gap_n)
    mask = np.zeros(len(times), dtype=bool)
    mask[start:end] = True
    return mask


def interpolate_mask(
    times: np.ndarray,
    signal: np.ndarray,
    mask: np.ndarray,
) -> tuple[dict[str, np.ndarray], np.ndarray]:
    observed = signal.astype(float).copy()
    observed[mask] = np.nan
    blink_df = mask_to_blinks_df(mask, times)
    preds = {}
    for method in METHODS:
        data = pd.DataFrame({"timestamp": times, "signal": observed})
        out = pipe.interpolate_blinks(
            data,
            blink_df,
            columns=["signal"],
            method=method,
            margin_ms=0.0,
        )
        preds[method] = out["signal"].to_numpy(dtype=float)
    return preds, observed


def score_methods(
    signal: np.ndarray,
    preds: dict[str, np.ndarray],
    mask: np.ndarray,
) -> pd.DataFrame:
    rows = []
    for method, pred in preds.items():
        err = pred[mask] - signal[mask]
        err = err[np.isfinite(err)]
        rows.append(
            {
                "method": method,
                "mae": np.nan if err.size == 0 else float(np.mean(np.abs(err))),
                "rmse": np.nan if err.size == 0 else float(np.sqrt(np.mean(err ** 2))),
                "n": int(err.size),
            }
        )
    return pd.DataFrame(rows).sort_values("rmse", na_position="last")


def plot_interpolation_window(
    times: np.ndarray,
    signal: np.ndarray,
    observed: np.ndarray,
    preds: dict[str, np.ndarray],
    mask: np.ndarray,
    title: str,
    signal_name: str,
    pad_s: float = 0.5,
) -> None:
    if not mask.any():
        print("Nothing to plot: mask is empty.")
        return

    gap_times = times[mask]
    t0, t1 = float(gap_times[0]), float(gap_times[-1])
    mid = 0.5 * (t0 + t1)
    z = (times >= mid - pad_s) & (times <= mid + pad_s)

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(times[z], signal[z], lw=1.2, alpha=0.65, label=f"original {signal_name}")
    ax.plot(times[z], observed[z], ".", ms=3, alpha=0.8, label="visible samples")
    for method, pred in preds.items():
        ax.plot(times[z], pred[z], lw=1.15, label=method)
    ax.axvspan(t0, t1, color="gold", alpha=0.18)
    ax.axvline(t0, color="0.5", ls="--", lw=1.0)
    ax.axvline(t1, color="0.5", ls="--", lw=1.0)
    ax.set_title(title)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel(signal_name)
    ax.legend(fontsize=8, ncol=3)
    plt.tight_layout()
    plt.show()


def plot_overview(times: np.ndarray, signal: np.ndarray, blinks: pd.DataFrame, signal_name: str) -> None:
    fig, ax = plt.subplots(figsize=(12, 3.5))
    ax.plot(times, signal, lw=0.8, alpha=0.75)
    if blinks is not None and not blinks.empty:
        for _, blink in blinks.head(40).iterrows():
            start = blink.get("start_timestamp", np.nan)
            end = blink.get("end_timestamp", start)
            if np.isfinite(start):
                if not np.isfinite(end) or end <= start:
                    end = start + 0.15
                ax.axvspan(start, end, color="gold", alpha=0.15)
    ax.set_title(f"Overview: {signal_name} with first blink intervals shaded")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel(signal_name)
    plt.tight_layout()
    plt.show()


## Inspect the Real Signal

Blink intervals are shaded. This first plot is deliberately broad: before choosing a method, look at the scale, smoothness, and missing-data pattern.


In [10]:
plot_overview(times, signal, blinks, signal_col)


/var/folders/n5/rzkmrrl9699cvlrds_01_ww00000gn/T/ipykernel_50577/3655933003.py:149: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Compare Methods on One Held-Out Gap

Here we hide a clean segment where the true signal is still known. This gives each method the same small challenge and lets us compute MAE/RMSE inside the hidden region.


In [11]:
heldout_mask = choose_heldout_mask(times, blinks, duration_s=HELDOUT_GAP_S)
heldout_preds, heldout_observed = interpolate_mask(times, signal, heldout_mask)
heldout_scores = score_methods(signal, heldout_preds, heldout_mask)

display(heldout_scores)

plot_interpolation_window(
    times,
    signal,
    heldout_observed,
    heldout_preds,
    heldout_mask,
    title=f"Held-out {HELDOUT_GAP_S * 1000:.0f} ms gap",
    signal_name=signal_col,
    pad_s=ZOOM_PAD_S,
)


,method,mae,rmse,n
3,weighted_average,0.032920,0.041815,40
0,linear,0.041636,0.050147,40
2,nearest,0.041708,0.050378,40
1,pchip,0.042252,0.050652,40
4,cubic,0.206335,0.221403,40


/var/folders/n5/rzkmrrl9699cvlrds_01_ww00000gn/T/ipykernel_50577/3655933003.py:131: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Inspect Native Blink Windows

Native blink windows do not have a ground-truth signal inside the blink. Use these plots for visual sanity checks: no wild jumps, no sharp overshoot, and a reasonable return to the surrounding signal.


In [12]:
if blinks.empty:
    print("No native blink intervals found in this recording.")
else:
    plotted = 0
    for _, blink in blinks.iterrows():
        if plotted >= N_NATIVE_BLINKS:
            break

        start_s = blink.get("start_timestamp", np.nan)
        end_s = blink.get("end_timestamp", start_s)
        if not np.isfinite(start_s):
            continue
        if not np.isfinite(end_s) or end_s <= start_s:
            end_s = start_s + 0.15

        blink_mask = (times >= start_s) & (times <= end_s)
        if blink_mask.sum() < 2:
            continue

        blink_preds, blink_observed = interpolate_mask(times, signal, blink_mask)
        plotted += 1
        print(f"Blink {plotted}: {1000 * (end_s - start_s):.0f} ms, {int(blink_mask.sum())} samples")
        plot_interpolation_window(
            times,
            signal,
            blink_observed,
            blink_preds,
            blink_mask,
            title=f"Native blink {plotted}",
            signal_name=signal_col,
            pad_s=ZOOM_PAD_S,
        )

    if plotted == 0:
        print("Blink intervals were present, but none overlapped enough samples to plot.")


Blink 1: 215 ms, 44 samples
Blink 2: 275 ms, 56 samples
Blink 3: 195 ms, 40 samples
Blink 4: 260 ms, 53 samples


/var/folders/n5/rzkmrrl9699cvlrds_01_ww00000gn/T/ipykernel_50577/3655933003.py:131: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Change the Parameters and Re-run

Try different values and compare the plot and scores:

- `GAP_DURATION_MS = 100`, `200`, `400`
- `SELECTED_METHOD = "linear"`, `"pchip"`, `"nearest"`, `"weighted_average"`

In [ ]:
GAP_DURATION_MS = 200
SELECTED_METHOD = "linear"

assert SELECTED_METHOD in METHODS, f"SELECTED_METHOD must be one of {METHODS}"

parameter_mask  = choose_heldout_mask(times, blinks, duration_s=GAP_DURATION_MS / 1000.0)
parameter_preds, parameter_observed = interpolate_mask(times, signal, parameter_mask)
selected_pred   = parameter_preds[SELECTED_METHOD]

display(score_methods(signal, {SELECTED_METHOD: selected_pred}, parameter_mask))
plot_interpolation_window(
    times, signal, parameter_observed,
    {SELECTED_METHOD: selected_pred},
    parameter_mask,
    title=f"{SELECTED_METHOD} on a {GAP_DURATION_MS} ms held-out gap",
    signal_name=signal_col,
    pad_s=ZOOM_PAD_S,
)

## Optional Interactive Playground

If `ipywidgets` is installed, use the sliders to explore gap duration and
interpolation method interactively. If not, edit variables in the cell above.

In [ ]:
try:
    import ipywidgets as widgets
except ImportError:
    print("ipywidgets is not installed — use the parameter cell above instead.")
else:
    gap_slider      = widgets.IntSlider(value=200, min=50, max=600, step=25, description="gap ms")
    method_dropdown = widgets.Dropdown(options=METHODS, value="linear", description="method")

    def _interactive_plot(gap_ms: int, method: str) -> None:
        mask = choose_heldout_mask(times, blinks, duration_s=gap_ms / 1000.0)
        preds, observed = interpolate_mask(times, signal, mask)
        display(score_methods(signal, {method: preds[method]}, mask))
        plot_interpolation_window(
            times, signal, observed,
            {method: preds[method]},
            mask,
            title=f"Interactive: {method}, {gap_ms} ms gap",
            signal_name=signal_col,
            pad_s=ZOOM_PAD_S,
        )

    out = widgets.interactive_output(
        _interactive_plot,
        {"gap_ms": gap_slider, "method": method_dropdown},
    )
    display(widgets.VBox([gap_slider, method_dropdown, out]))

## Short Reflection

Answer these in a markdown cell below, or discuss them as a group:

1. Which method had the lowest RMSE on the held-out segment?
2. Did the lowest-RMSE method also look best around native blinks?
3. What changed when you increased the gap duration from 100 ms to 400 ms?
4. Would you ever reject a trial entirely instead of interpolating it? At what gap duration?
5. Why does `nearest` have low error in the table but look bad near the gap boundaries?